In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
from qiskit_aer import AerSimulator
import math

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

From BB84-Plain.ipynb
1. quantum_random_bit()
2. encode_qubit()
3. measure_qubit()
4. Alice's section (bits, bases, qubits)

In [14]:

simulator = AerSimulator()

def quantum_random_bit():
    qc = QuantumCircuit(1, 1)
    qc.h(0)           # puts qubit into (|0⟩ + |1⟩)/√2
    qc.measure(0, 0)
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

def encode_qubit(bit, basis):
    qc = QuantumCircuit(1, 1)
    
    if bit == 1:
        qc.x(0)        # flip to |1⟩
    
    if basis == 1:
        qc.h(0)        # rotate to diagonal basis (× )
    
    return qc

def measure_qubit(qc, basis):
    if basis == 1:
        qc.h(0)        # rotate to diagonal basis before measuring
    
    qc.measure(0, 0)
    
    job = simulator.run(qc, shots=1)
    result = job.result()
    counts = result.get_counts()
    return int(list(counts.keys())[0])

# Alice 
n = 100
alice_bits  = [quantum_random_bit() for _ in range(n)]
alice_bases = [quantum_random_bit() for _ in range(n)]
alice_qubits = [encode_qubit(alice_bits[i], alice_bases[i]) for i in range(n)]



Attacker Eve : sits between Alice and Bob. intercepts each qubit, measure it then resends a new qubit to Bob based on what she measured. 

Attack 1 Full intercept: intercepts and measure every qubit

In [15]:
eve_bases = [quantum_random_bit() for _ in range(n)]
eve_results = []
intercepted_qubits = []

for i in range(n):
    # Eve measures Alice's qubit in her random basis
    eve_bit = measure_qubit(alice_qubits[i], eve_bases[i])
    eve_results.append(eve_bit)
    
    # Eve resends a new qubit to Bob based on what she measured
    new_qubit = encode_qubit(eve_bit, eve_bases[i])
    intercepted_qubits.append(new_qubit)
    
print(f"Eve intercepted all {n} qubits")

Eve intercepted all 100 qubits


In [16]:
# Bob
bob_bases = [quantum_random_bit() for _ in range(n)]
bob_results = [measure_qubit(intercepted_qubits[i], bob_bases[i]) for i in range(n)]

In [17]:
# Basis sifting

sifted_alice = []
sifted_bob = []
sifted_indices = []

for i in range(n):
    if alice_bases[i] == bob_bases[i]:
        sifted_alice.append(alice_bits[i])
        sifted_bob.append(bob_results[i])
        sifted_indices.append(i)

print(f"Total qubits sent:     {n}")
print(f"Sifted key length:     {len(sifted_alice)}")
print(f"Alice's sifted key:    {sifted_alice}")
print(f"Bob's sifted key:      {sifted_bob}")

Total qubits sent:     100
Sifted key length:     49
Alice's sifted key:    [0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1]
Bob's sifted key:      [1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1]


Error Detection to detect attacker 

In [18]:
# Sample 25% of the sifted key to check for errors
sample_size = len(sifted_alice) // 4

errors = 0
for i in range(sample_size):
    if sifted_alice[i] != sifted_bob[i]:
        errors += 1

error_rate = errors / sample_size
threshold = 0.1  # 10% threshold

print(f"Sample size:           {sample_size}")
print(f"Errors found:          {errors}")
print(f"Error rate:            {error_rate:.2%}")
print(f"Threshold:             {threshold:.2%}")
print()
if error_rate > threshold:
    print("ATTACK DETECTED: error rate too high!")
else:
    print("No attack detected.")

Sample size:           12
Errors found:          2
Error rate:            16.67%
Threshold:             10.00%

ATTACK DETECTED: error rate too high!


Attack 2 Partial Attack: only intercepts 50% of qubits 

In [19]:
# Fresh run for Alice
alice_bits_2   = [quantum_random_bit() for _ in range(n)]
alice_bases_2  = [quantum_random_bit() for _ in range(n)]
alice_qubits_2 = [encode_qubit(alice_bits_2[i], alice_bases_2[i]) for i in range(n)]



In [20]:
# Eve partial attack
eve_bases_2 = [quantum_random_bit() for _ in range(n)]
eve_results_2 = []
intercepted_qubits_2 = []
intercept_rate = 0.5

for i in range(n):
    if quantum_random_bit() == 0:  # ~50% chance
        # Eve intercepts and resends
        eve_bit = measure_qubit(alice_qubits_2[i], eve_bases_2[i])
        eve_results_2.append(eve_bit)
        new_qubit = encode_qubit(eve_bit, eve_bases_2[i])
        intercepted_qubits_2.append(new_qubit)
    else:
        # Eve lets qubit through untouched
        eve_results_2.append(None)
        intercepted_qubits_2.append(alice_qubits_2[i])

print(f"Eve intercepted {sum(1 for e in eve_results_2 if e is not None)} out of {n} qubits")


Eve intercepted 44 out of 100 qubits


In [21]:
# Bob
bob_bases_2   = [quantum_random_bit() for _ in range(n)]
bob_results_2 = [measure_qubit(intercepted_qubits_2[i], bob_bases_2[i]) for i in range(n)]

In [ ]:
# Basis Sifting 
sifted_alice_2 = []
sifted_bob_2   = []

for i in range(n):
    if alice_bases_2[i] == bob_bases_2[i]:
        sifted_alice_2.append(alice_bits_2[i])
        sifted_bob_2.append(bob_results_2[i])

print(f"Total qubits sent:     {n}")
print(f"Sifted key length:     {len(sifted_alice_2)}")
print(f"Alice's sifted key:    {sifted_alice_2}")
print(f"Bob's sifted key:      {sifted_bob_2}")

In [ ]:
# Error Checking 
sample_size_2 = len(sifted_alice_2) // 4
errors_2 = 0

for i in range(sample_size_2):
    if sifted_alice_2[i] != sifted_bob_2[i]:
        errors_2 += 1

error_rate_2 = errors_2 / sample_size_2
threshold = 0.1

print("ATTACK 2: PARTIAL INTERCEPT (50%)")
print(f"Sample size:           {sample_size_2}")
print(f"Errors found:          {errors_2}")
print(f"Error rate:            {error_rate_2:.2%}")
print(f"Threshold:             {threshold:.2%}")
print()
if error_rate_2 > threshold:
    print("ATTACK DETECTED: error rate too high!")
else:
    print("No attack detected: Eve may have slipped through!")

--- ATTACK 2: PARTIAL INTERCEPT (50%) ---
Sample size:           13
Errors found:          0
Error rate:            0.00%
Threshold:             10.00%

No attack detected: Eve may have slipped through!


Why partial interception can escape detection

When Eve intercepts only 50% of qubits, she introduces errors on roughly half the qubits she touches. she randomly chooses her measurement basis, she guesses correctly ~50% of the time so intercepted qubits cause errors with probability 1/4 (wrong basis AND wrong basis for Bob). With 50% interception, the expected error rate is ~12.5%, which is above the 10% threshold on average. However, with a small sample (≈13 bits here), there is meaningful variance: it is quite possible by chance that few or no errors fall in the sample window, causing detection to fail. A larger sample size or a lower threshold would increase the detection probability.